# Q3 genotype QC and LD

**Question:** What do genotype completeness, HWE, coding/harmonization status and LD permit us to conclude?

**Inputs:** Validated genotype-wide file; optional supplied genetic QC/ancestry/unrelatedness and allele-harmonization files.

**Outputs:** REF/ALT coding records, harmonization status, pooled/available ancestry-specific QC and LD summaries.

Broad environment/schema searches and literature task lists are omitted. Their removal is not evidence that PCs, relatedness adjustment or allele harmonization succeeded. Inspect the supplied-input statuses. Pooled HWE is descriptive; unphased EM D-prime is an estimate, not observed phasing.


## Configuration


In [ ]:
import itertools
import math
import os
import re
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from google.cloud import bigquery
except Exception:
    bigquery = None

try:
    from scipy.stats import chi2
except Exception:
    chi2 = None

RUN_ID = os.environ.get('Q3_VDR_RUN_ID', pd.Timestamp.utcnow().strftime('%Y%m%dT%H%M%SZ'))
TIER1_SNPS = [
    {'rsid': 'rs731236', 'label': 'TaqI', 'contig': 'chr12', 'position': 47844974, 'ref': 'A', 'alt': 'G'},
    {'rsid': 'rs7975232', 'label': 'ApaI', 'contig': 'chr12', 'position': 47845054, 'ref': 'C', 'alt': 'A'},
    {'rsid': 'rs1544410', 'label': 'BsmI', 'contig': 'chr12', 'position': 47846052, 'ref': 'C', 'alt': 'T'},
    {'rsid': 'rs2228570', 'label': 'FokI', 'contig': 'chr12', 'position': 47879112, 'ref': 'A', 'alt': 'G'},
]
TIER1_RSIDS = [x['rsid'] for x in TIER1_SNPS]

def infer_default_data_dir():
    cwd = Path.cwd()
    candidates = [
        cwd,
        Path('/home/dataproc/workspace') / 'VDR',
        Path('/home/jupyter/workspace') / 'VDR',
        cwd.parent,
    ]
    for candidate in candidates:
        try:
            if candidate.exists() and os.access(candidate, os.W_OK):
                return candidate
        except Exception:
            pass
    return cwd

DATA_DIR = Path(os.environ.get('Q3_VDR_DATA_DIR', str(infer_default_data_dir()))).expanduser()
OUTPUT_DIR = Path(os.environ.get('Q3_VDR_STEP5_VALIDATION_OUTPUT_DIR', str(DATA_DIR / 'Q3_VDR_step5_remaining_validation_outputs_2026_08_09'))).expanduser()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def split_project_dataset(value):
    value = str(value).strip().strip('`')
    if '.' in value:
        return value.split('.', 1)
    return '', value

CDR = os.environ.get('WORKSPACE_CDR', 'YOUR_GOOGLE_PROJECT.C2025Q4R6').strip()
CDR_PROJECT, CDR_DATASET = split_project_dataset(CDR)
PREP_CDR = os.environ.get('WORKSPACE_PREP_CDR', f'{CDR_PROJECT}.prep_{CDR_DATASET}' if CDR_PROJECT and not CDR_DATASET.startswith('prep_') else '').strip()

def save(df, filename):
    path = OUTPUT_DIR / filename
    df.to_csv(path, index=False)
    print(f'Saved {filename}: {len(df):,} rows')
    return path

def save_text(text, filename):
    path = OUTPUT_DIR / filename
    path.write_text(str(text), encoding='utf-8')
    print(f'Saved {filename}: {len(str(text)):,} chars')
    return path

def find_first_existing(names, dirs):
    for directory in dirs:
        directory = Path(directory)
        for name in names:
            path = directory / name
            if path.exists():
                return path
    return None

candidate_dirs = [
    DATA_DIR,
    DATA_DIR / 'Q3_VDR_tier1_hail_genotype_outputs_2026_08_07',
    DATA_DIR / 'Q3_VDR_tier1_additive_genotype_outputs_2026_08_08',
    DATA_DIR / 'Q3_VDR_step5_remaining_validation_outputs_2026_08_09',
]
env_genotype = os.environ.get('Q3_VDR_GENOTYPE_WIDE_CSV', '').strip()
GENOTYPE_WIDE_PATH = Path(env_genotype).expanduser() if env_genotype else find_first_existing(['14_tier1_vdr_genotype_wide.csv'], candidate_dirs)
env_qc = os.environ.get('Q3_VDR_GENETIC_QC_CSV', '').strip()
GENETIC_QC_PATH = Path(env_qc).expanduser() if env_qc else None
env_harmonization = os.environ.get('Q3_VDR_ALLELE_HARMONIZATION_CSV', '').strip()
ALLELE_HARMONIZATION_PATH = Path(env_harmonization).expanduser() if env_harmonization else None

status_rows = []
def add_status(step, status, detail=''):
    status_rows.append({'run_id': RUN_ID, 'step': step, 'status': status, 'detail': str(detail)[:4000]})
    save(pd.DataFrame(status_rows), '17_step5_validation_status.csv')

config = pd.DataFrame([{
    'run_id': RUN_ID,
    'data_dir': str(DATA_DIR),
    'output_dir': str(OUTPUT_DIR),
    'cdr': CDR,
    'prep_cdr': PREP_CDR,
    'genotype_wide_path': str(GENOTYPE_WIDE_PATH) if GENOTYPE_WIDE_PATH else '',
    'genetic_qc_path': str(GENETIC_QC_PATH) if GENETIC_QC_PATH else '',
    'allele_harmonization_path': str(ALLELE_HARMONIZATION_PATH) if ALLELE_HARMONIZATION_PATH else '',
}])
save(config, '17_step5_validation_run_configuration.csv')


## Record allele coding and supplied evidence


In [ ]:
aou_records = pd.DataFrame([
    {
        'run_id': RUN_ID,
        'rsid': x['rsid'],
        'label': x['label'],
        'aou_contig': x['contig'],
        'aou_grch38_position': x['position'],
        'aou_ref': x['ref'],
        'aou_alt': x['alt'],
        'aou_coding': 'ALT dosage = GT.n_alt_alleles() from AoU v9 WGS split Hail MT',
        'directional_status': 'not_harmonized_to_literature_effect_or_function_allele',
    }
    for x in TIER1_SNPS
])
save(aou_records, '17_tier1_vdr_aou_ref_alt_records_for_harmonization.csv')


harm_status = []
if ALLELE_HARMONIZATION_PATH and ALLELE_HARMONIZATION_PATH.exists():
    h = pd.read_csv(ALLELE_HARMONIZATION_PATH)
    for rsid in TIER1_RSIDS:
        sub = h[h.astype(str).apply(lambda row: row.str.contains(rsid, case=False, regex=False).any(), axis=1)]
        status = 'provided_review_required' if not sub.empty else 'missing_for_rsid'
        harm_status.append({'run_id': RUN_ID, 'rsid': rsid, 'status': status, 'evidence': str(ALLELE_HARMONIZATION_PATH)})
else:
    for rsid in TIER1_RSIDS:
        harm_status.append({'run_id': RUN_ID, 'rsid': rsid, 'status': 'not_provided', 'evidence': 'Set Q3_VDR_ALLELE_HARMONIZATION_CSV after manual dbSNP/Ensembl/literature harmonization.'})
save(pd.DataFrame(harm_status), '17_allele_harmonization_gate_status.csv')
add_status('allele_harmonization_package', 'pass', 'AoU REF/ALT coding and supplied harmonization evidence recorded')


## Load genotype and optional genetic QC


In [ ]:
genotype_status = []
geno = pd.DataFrame()
if GENOTYPE_WIDE_PATH and GENOTYPE_WIDE_PATH.exists():
    geno = pd.read_csv(GENOTYPE_WIDE_PATH)
    genotype_status.append({'run_id': RUN_ID, 'input': 'genotype_wide', 'status': 'loaded', 'path': str(GENOTYPE_WIDE_PATH), 'rows': len(geno), 'columns': len(geno.columns)})
else:
    genotype_status.append({'run_id': RUN_ID, 'input': 'genotype_wide', 'status': 'missing', 'path': str(GENOTYPE_WIDE_PATH) if GENOTYPE_WIDE_PATH else '', 'rows': 0, 'columns': 0})

if not geno.empty:
    if 'person_id' not in geno.columns and 'sample_id' in geno.columns:
        geno['person_id'] = pd.to_numeric(geno['sample_id'], errors='coerce')
    for rsid in TIER1_RSIDS:
        if rsid in geno.columns and f'{rsid}_alt_dosage' not in geno.columns:
            geno[f'{rsid}_alt_dosage'] = pd.to_numeric(geno[rsid], errors='coerce')
        elif f'{rsid}_alt_dosage' in geno.columns:
            geno[f'{rsid}_alt_dosage'] = pd.to_numeric(geno[f'{rsid}_alt_dosage'], errors='coerce')
    keep = ['person_id', 'sample_id'] + [c for c in [f'{rsid}_alt_dosage' for rsid in TIER1_RSIDS] if c in geno.columns]
    geno = geno[[c for c in keep if c in geno.columns]].copy()
save(pd.DataFrame(genotype_status), '17_genotype_wide_load_status.csv')

qc = pd.DataFrame()
qc_status = []
if GENETIC_QC_PATH and GENETIC_QC_PATH.exists():
    qc = pd.read_csv(GENETIC_QC_PATH)
    qc_status.append({'run_id': RUN_ID, 'input': 'genetic_qc', 'status': 'loaded', 'path': str(GENETIC_QC_PATH), 'rows': len(qc), 'columns': len(qc.columns)})
else:
    qc_status.append({'run_id': RUN_ID, 'input': 'genetic_qc', 'status': 'not_provided', 'path': str(GENETIC_QC_PATH) if GENETIC_QC_PATH else '', 'rows': 0, 'columns': 0})

joined = geno.copy()
join_status = 'no_qc_join'
if not geno.empty and not qc.empty:
    join_key = None
    for key in ['person_id', 'sample_id']:
        if key in geno.columns and key in qc.columns:
            join_key = key
            break
    if join_key:
        joined = geno.merge(qc, on=join_key, how='left', validate='many_to_one')
        join_status = f'joined_on_{join_key}'
    else:
        join_status = 'qc_loaded_no_common_person_or_sample_key'
qc_status.append({'run_id': RUN_ID, 'input': 'genotype_qc_join', 'status': join_status, 'path': '', 'rows': len(joined), 'columns': len(joined.columns)})
save(pd.DataFrame(qc_status), '17_genetic_qc_load_join_status.csv')

def find_cols(patterns, cols):
    hits = []
    for col in cols:
        low = col.lower()
        if any(re.search(p, low) for p in patterns):
            hits.append(col)
    return hits

pc_cols = find_cols([r'^pc\d+$', r'principal.*component', r'genetic.*pc', r'ancestry.*pc'], joined.columns)
ancestry_cols = find_cols([r'genetic.*ancestry', r'inferred.*ancestry', r'predicted.*ancestry', r'ancestry_group', r'population'], joined.columns)
related_cols = find_cols([r'unrelated', r'kinship', r'relatedness', r'related_pair', r'relationship'], joined.columns)
sample_qc_cols = find_cols([r'sample.*qc', r'qc.*pass', r'call.*rate', r'missingness', r'hard.*filter'], joined.columns)
column_hint_rows = []
for domain, cols in [('pc', pc_cols), ('ancestry', ancestry_cols), ('relatedness_unrelated', related_cols), ('sample_qc', sample_qc_cols)]:
    column_hint_rows.append({'run_id': RUN_ID, 'domain': domain, 'candidate_columns': ';'.join(cols), 'n_columns': len(cols)})
save(pd.DataFrame(column_hint_rows), '17_joined_genetic_qc_column_hints.csv')

add_status('load_genotype_and_optional_qc', 'pass' if not geno.empty else 'missing_genotype', f'geno_rows={len(geno)}; qc_rows={len(qc)}; join={join_status}')


## Genotype and HWE summaries


In [ ]:
def pct(n, d):
    return round(float(n) * 100.0 / float(d), 3) if d else np.nan

def hwe_summary(x):
    x = pd.to_numeric(x, errors='coerce')
    x = x[x.isin([0, 1, 2])]
    n0 = int((x == 0).sum())
    n1 = int((x == 1).sum())
    n2 = int((x == 2).sum())
    n = n0 + n1 + n2
    if n == 0:
        return {'n_called': 0, 'genotype_0': 0, 'genotype_1': 0, 'genotype_2': 0, 'alt_af': np.nan, 'maf': np.nan, 'hwe_chisq': np.nan, 'hwe_p_chisq': np.nan}
    p_alt = (n1 + 2 * n2) / (2 * n)
    exp0 = n * (1 - p_alt) ** 2
    exp1 = n * 2 * p_alt * (1 - p_alt)
    exp2 = n * p_alt ** 2
    chisq = sum((obs - exp) ** 2 / exp for obs, exp in [(n0, exp0), (n1, exp1), (n2, exp2)] if exp > 0)
    hwe_p = float(chi2.sf(chisq, 1)) if chi2 is not None and np.isfinite(chisq) else np.nan
    return {'n_called': n, 'genotype_0': n0, 'genotype_1': n1, 'genotype_2': n2, 'alt_af': p_alt, 'maf': min(p_alt, 1 - p_alt), 'hwe_chisq': chisq, 'hwe_p_chisq': hwe_p}

qc_rows = []
if not joined.empty:
    for rsid in TIER1_RSIDS:
        dosage = f'{rsid}_alt_dosage'
        if dosage not in joined.columns:
            qc_rows.append({'run_id': RUN_ID, 'scope': 'pooled_all_loaded_samples', 'group': 'all', 'rsid': rsid, 'status': 'missing_dosage_column'})
            continue
        x = joined[dosage]
        s = hwe_summary(x)
        row = {'run_id': RUN_ID, 'scope': 'pooled_all_loaded_samples', 'group': 'all', 'rsid': rsid, 'status': 'descriptive_only_unrelated_ancestry_not_required_here', 'n_total': len(joined), 'n_missing': int(pd.to_numeric(x, errors='coerce').isna().sum()), 'missing_pct': pct(int(pd.to_numeric(x, errors='coerce').isna().sum()), len(joined))}
        row.update(s)
        qc_rows.append(row)

pooled_qc = pd.DataFrame(qc_rows)
save(pooled_qc, '17_tier1_vdr_pooled_genotype_qc_hwe_descriptive.csv')

ancestry_col = os.environ.get('Q3_VDR_ANCESTRY_COL', '').strip()
unrelated_col = os.environ.get('Q3_VDR_UNRELATED_COL', '').strip()
if not ancestry_col and ancestry_cols:
    ancestry_col = ancestry_cols[0]
if not unrelated_col and related_cols:
    unrelated_candidates = [c for c in related_cols if 'unrelated' in c.lower()]
    unrelated_col = unrelated_candidates[0] if unrelated_candidates else ''

hwe_group_rows = []
hwe_gate_status = 'not_available_no_joined_ancestry_unrelated'
if not joined.empty and ancestry_col in joined.columns and unrelated_col in joined.columns:
    u = joined[unrelated_col]
    unrelated_mask = u.astype(str).str.lower().isin(['1', 'true', 'yes', 'y', 'unrelated', 'pass']) | (pd.to_numeric(u, errors='coerce') == 1)
    d = joined[unrelated_mask].copy()
    if len(d) > 0:
        for group, sub in d.groupby(ancestry_col, dropna=False):
            for rsid in TIER1_RSIDS:
                dosage = f'{rsid}_alt_dosage'
                if dosage not in sub.columns:
                    continue
                s = hwe_summary(sub[dosage])
                row = {'run_id': RUN_ID, 'scope': 'unrelated_by_ancestry', 'ancestry_col': ancestry_col, 'unrelated_col': unrelated_col, 'ancestry_group': str(group), 'rsid': rsid, 'n_total': len(sub), 'status': 'computed'}
                row.update(s)
                hwe_group_rows.append(row)
        hwe_gate_status = 'computed_unrelated_by_ancestry'
    else:
        hwe_gate_status = 'unrelated_col_found_but_no_unrelated_rows_detected'
else:
    hwe_group_rows.append({'run_id': RUN_ID, 'scope': 'unrelated_by_ancestry', 'status': hwe_gate_status, 'ancestry_col': ancestry_col, 'unrelated_col': unrelated_col})
save(pd.DataFrame(hwe_group_rows), '17_tier1_vdr_hwe_by_unrelated_ancestry_if_available.csv')
save(pd.DataFrame([{'run_id': RUN_ID, 'hwe_by_unrelated_ancestry_status': hwe_gate_status, 'ancestry_col': ancestry_col, 'unrelated_col': unrelated_col, 'note': 'Pooled HWE is descriptive only unless unrelated ancestry grouping is available.'}]), '17_hwe_unrelated_ancestry_gate_status.csv')
add_status('hwe_checks', 'pass' if not pooled_qc.empty else 'skip_no_genotype', hwe_gate_status)


## Pairwise LD and unphased haplotype estimates


In [ ]:
def possible_diplotypes(g1, g2):
    haps = [(0, 0), (0, 1), (1, 0), (1, 1)]
    pairs = []
    for i, h1 in enumerate(haps):
        for j, h2 in enumerate(haps[i:], start=i):
            if h1[0] + h2[0] == g1 and h1[1] + h2[1] == g2:
                pairs.append((i, j))
    return pairs

def em_pairwise_ld(g1, g2, max_iter=500, tol=1e-10):
    dat = pd.DataFrame({'g1': pd.to_numeric(g1, errors='coerce'), 'g2': pd.to_numeric(g2, errors='coerce')}).dropna()
    dat = dat[dat['g1'].isin([0, 1, 2]) & dat['g2'].isin([0, 1, 2])]
    n = len(dat)
    if n == 0:
        return {'n_pair_called': 0, 'r2_corr': np.nan, 'd_prime_em': np.nan, 'r2_em': np.nan, 'D_em': np.nan, 'p_alt_1': np.nan, 'p_alt_2': np.nan}
    p1 = float(dat['g1'].sum() / (2 * n))
    p2 = float(dat['g2'].sum() / (2 * n))
    freqs = np.array([(1 - p1) * (1 - p2), (1 - p1) * p2, p1 * (1 - p2), p1 * p2], dtype=float)
    freqs = np.clip(freqs, 1e-6, None)
    freqs = freqs / freqs.sum()
    diplotypes = [possible_diplotypes(int(a), int(b)) for a, b in dat[['g1', 'g2']].itertuples(index=False)]
    for _ in range(max_iter):
        counts = np.zeros(4, dtype=float)
        for pairs in diplotypes:
            probs = []
            for i, j in pairs:
                prob = freqs[i] * freqs[j] * (2.0 if i != j else 1.0)
                probs.append(prob)
            total = sum(probs)
            if total <= 0:
                continue
            for (i, j), prob in zip(pairs, probs):
                w = prob / total
                counts[i] += w
                counts[j] += w
        new_freqs = counts / counts.sum()
        if np.max(np.abs(new_freqs - freqs)) < tol:
            freqs = new_freqs
            break
        freqs = new_freqs
    h00, h01, h10, h11 = freqs
    pA = h10 + h11
    pB = h01 + h11
    D = h11 - pA * pB
    denom = pA * (1 - pA) * pB * (1 - pB)
    r2_em = (D * D / denom) if denom > 0 else np.nan
    if D >= 0:
        Dmax = min(pA * (1 - pB), (1 - pA) * pB)
    else:
        Dmax = min(pA * pB, (1 - pA) * (1 - pB))
    dprime = D / Dmax if Dmax > 0 else np.nan
    corr = dat['g1'].corr(dat['g2'])
    return {
        'n_pair_called': n,
        'r2_corr': corr ** 2 if pd.notna(corr) else np.nan,
        'd_prime_em': dprime,
        'r2_em': r2_em,
        'D_em': D,
        'p_alt_1': pA,
        'p_alt_2': pB,
        'hap_00_ref_ref': h00,
        'hap_01_ref_alt2': h01,
        'hap_10_alt1_ref': h10,
        'hap_11_alt_alt': h11,
    }

ld_rows = []
if not joined.empty:
    for rsid1, rsid2 in itertools.combinations(TIER1_RSIDS, 2):
        c1 = f'{rsid1}_alt_dosage'
        c2 = f'{rsid2}_alt_dosage'
        if c1 not in joined.columns or c2 not in joined.columns:
            ld_rows.append({'run_id': RUN_ID, 'snp1': rsid1, 'snp2': rsid2, 'status': 'missing_dosage_column'})
            continue
        row = {'run_id': RUN_ID, 'snp1': rsid1, 'snp2': rsid2, 'status': 'computed_unphased_em'}
        row.update(em_pairwise_ld(joined[c1], joined[c2]))
        ld_rows.append(row)
else:
    ld_rows.append({'run_id': RUN_ID, 'snp1': '', 'snp2': '', 'status': 'skip_no_genotype'})
ld_df = pd.DataFrame(ld_rows)
save(ld_df, '17_tier1_vdr_pairwise_ld_r2_dprime_unphased_em.csv')

three_prime = ['rs731236', 'rs7975232', 'rs1544410']
complete_n = 0
if not joined.empty and all(f'{x}_alt_dosage' in joined.columns for x in three_prime):
    complete_n = int(joined[[f'{x}_alt_dosage' for x in three_prime]].dropna().shape[0])
hap_status = pd.DataFrame([{
    'run_id': RUN_ID,
    'haplotype_set': 'VDR_3prime_rs731236_rs7975232_rs1544410',
    'required_snps': ';'.join(three_prime),
    'n_complete_unphased_genotypes': complete_n,
    'pairwise_dprime_available': bool((ld_df['status'] == 'computed_unphased_em').any()) if 'status' in ld_df.columns else False,
    'phased_haplotype_available': False,
    'status': 'pairwise_dprime_estimated_from_unphased_genotypes_only' if complete_n > 0 else 'missing_genotype',
    'note': 'This does not replace phased haplotype analysis. Use only for LD caution unless a phased resource or validated haplotype method is added.',
}])
save(hap_status, '17_tier1_vdr_haplotype_feasibility_status.csv')
add_status('ld_dprime_haplotype_checks', 'pass' if not ld_df.empty else 'skip', hap_status.loc[0, 'status'])
